In [28]:
import os
from tqdm import tqdm
from dotenv import load_dotenv


load_dotenv()
MY_HUGGIEFACE_TOKEN = os.getenv("MY_HUGGIEFACE_TOKEN")

In [18]:
import pandas as pd

df = pd.read_csv("./data/raw/bbq_plaza_reviews.csv")


In [19]:
# test_df = df.head(20).copy()
test_df = df
test_df['text']

0     Dropping by this restaurant while shopping in ...
1     This was my very first time trying Thai-style ...
2     บาร์บีคิวพลาซ่าสาขาเซ็นทรัลเวิลด์\nทุกอย่างอร่...
3     ไม่ได้กินบาร์บีก้อนนานแล้ว วันนี้เห็นมี lunch ...
4     Review: 🇹🇭 | 🇬🇧\n\nส่วนตัวไม่ค่อยได้มากิน BarB...
                            ...                        
95    Overpriced and small portions as compared to p...
96    ทานที่นี่ดีกว่าไปทานในห้างค่ะ คนไม่เยอะ แถมนั่...
97    Loved the buffet! Want to go back for more. Be...
98    รายการโปร ไม่รับ walk in ต้องจองล่วงหน้า แต่ไม...
99                                               ดีมากๆ
Name: text, Length: 100, dtype: object

# STEP 1: Sentimental Analysis

In [20]:
from transformers import pipeline

# Initialize the pipeline for sentiment analysis
# It will automatically download the model files shown in your screenshot
# Token indices sequence length is longer than the specified maximum sequence length for this model (603 > 510). Running this sequence through the model will result in indexing errors
classifier = pipeline("sentiment-analysis", model="SiemonCha/thai-sentiment-phayabert")

Device set to use mps:0


In [21]:
def predict_long_text(text, classifier, max_len=500):
	# 1. Split text into chunks of 500 characters (approx tokens)
	# Thai doesn't use spaces, so character splitting is safer than word splitting
	chunks = [text[i:i+max_len] for i in range(0, len(text), max_len)]
	
	results = []
	for chunk in chunks:
		# Run classifier on this specific chunk
		res = classifier(chunk, truncation=True, max_length=512)[0]
		results.append(res)
	
	# 2. Logic to combine results
	# Strategy: If ANY chunk is 'neg' (negative), consider the whole thing negative.
	# Otherwise, take the one with the highest confidence score.
	
	# Check if any chunk is negative
	neg_results = [r for r in results if r['label'] == 'neg']
	if neg_results:
		# Return the strongest negative result
		return max(neg_results, key=lambda x: x['score'])
	
	# Otherwise return the strongest result overall
	return max(results, key=lambda x: x['score'])

In [29]:
labels = {'LABEL_0': "Positive", 'LABEL_1': "Neutral", 'LABEL_2': "Negative"}

for index, row in tqdm(test_df.iterrows()):
	text = row['text']
	result = predict_long_text(text, classifier)
	label = labels[result["label"]]
	score = result["score"]
	# print(text)
	# print(f"[Result: {label}, Score: {score}]\n")
	test_df.loc[index, 'sentiment'] = label
	test_df.loc[index, 'sentiment_score'] = score

100it [00:22,  4.40it/s]


In [23]:
test_df

,review_id,author,rating,text,sentiment,sentiment_score
0,Ci9DQUlRQUNvZENodHljRjlvT2pkTk1URTNORTB4VXpaM1...,Tuy Chanmonypech,4,Dropping by this restaurant while shopping in ...,Positive,0.772329
1,Ci9DQUlRQUNvZENodHljRjlvT2t0SFUyaENablZ1UTNGTV...,李志東,4,This was my very first time trying Thai-style ...,Positive,0.998169
2,Ci9DQUlRQUNvZENodHljRjlvT2tkVlgwZHdlR1J3YlZobl...,Park Bodhisundara,5,บาร์บีคิวพลาซ่าสาขาเซ็นทรัลเวิลด์\nทุกอย่างอร่...,Positive,0.999520
3,Ci9DQUlRQUNvZENodHljRjlvT2tab09XWkdjR0pyWTBkMm...,Princess Mia,3,ไม่ได้กินบาร์บีก้อนนานแล้ว วันนี้เห็นมี lunch ...,Negative,0.999632
4,ChdDSUhNMG9nS0VJQ0FnSURmcjRPMnNnRRAB,Jira Tong,5,Review: 🇹🇭 | 🇬🇧\n\nส่วนตัวไม่ค่อยได้มากิน BarB...,Neutral,0.997700
...,...,...,...,...,...,...
95,ChZDSUhNMG9nS0VJQ0FnSURJMXJERUN3EAE,Camila K,3,Overpriced and small portions as compared to p...,Negative,0.999206
96,ChdDSUhNMG9nS0VJQ0FnSUNVX0kzdzhnRRAB,Kussjung Love,4,ทานที่นี่ดีกว่าไปทานในห้างค่ะ คนไม่เยอะ แถมนั่...,Positive,0.998901
97,ChZDSUhNMG9nS0VJQ0FnSURzbk1maENREAE,Karina Suvapataya,5,Loved the buffet! Want to go back for more. Be...,Neutral,0.908327
98,ChdDSUhNMG9nS0VJQ0FnSUQybXYtQ3RBRRAB,ยอด สุดสวย,1,รายการโปร ไม่รับ walk in ต้องจองล่วงหน้า แต่ไม...,Negative,0.999191


More manual

In [24]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch

# model = AutoModelForSequenceClassification.from_pretrained("SiemonCha/thai-sentiment-phayabert")
# tokenizer = AutoTokenizer.from_pretrained("SiemonCha/thai-sentiment-phayabert")

# reviews = [
#     "ร้านนี้อร่อยมาก บริการดีสุดๆ",          # ควรได้ positive
#     "รอนานมาก อาหารก็เย็นชืด ไม่ไหวเลย",   # ควรได้ negative
#     "ร้านเปิดกี่โมงครับ",                    # ควรได้ neutral (หรือ q)
#     "รสชาติงั้นๆ เฉยๆ พอกินได้"                  # ควรได้ neutral
# ]

# for text in reviews:
# 	inputs = tokenizer(text, return_tensors="pt")
# 	outputs = model(**inputs)
# 	prediction = torch.argmax(outputs.logits, dim=-1).item()

# 	labels = {0: "positive", 1: "neutral", 2: "negative"}
# 	print(labels[prediction])  # positive

---

# STEP 2: Zero-shot-classification

multi_label=True: It runs a Sigmoid function on each label's "Entailment" vs. "Contradiction" logit.
Each label gets a score between 0 and 1 independent of the others.

In [25]:
classifier_th = pipeline(
	"zero-shot-classification",
	model="joeddav/xlm-roberta-large-xnli" # Supports Thai
)

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [30]:
candidate_labels = ["Taste", "Portion", "Price", "Quality", "Service", "Speed", "Location"]
ACCEPTABLE_SCORE = 0.7

for index, row in tqdm(test_df.iterrows()):
	text = row['text']
	result = classifier_th(text, candidate_labels, multi_label=True)
	# print(f"Text: {text}")
	for label, score in zip(result['labels'], result['scores']):
		if score > ACCEPTABLE_SCORE:
			# print(f"{label}: {score:.4f}")
			test_df.loc[index, label] = 1
		else:
			test_df.loc[index, label] = 0

100it [08:21,  5.01s/it]


In [31]:
test_df

,review_id,author,rating,text,sentiment,sentiment_score,Service,Quality,Price,Taste,Location,Portion,Speed
0,Ci9DQUlRQUNvZENodHljRjlvT2pkTk1URTNORTB4VXpaM1...,Tuy Chanmonypech,4,Dropping by this restaurant while shopping in ...,Positive,0.772329,1.0,1.0,1.0,0.0,0.0,0.0,0.0
1,Ci9DQUlRQUNvZENodHljRjlvT2t0SFUyaENablZ1UTNGTV...,李志東,4,This was my very first time trying Thai-style ...,Positive,0.998169,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,Ci9DQUlRQUNvZENodHljRjlvT2tkVlgwZHdlR1J3YlZobl...,Park Bodhisundara,5,บาร์บีคิวพลาซ่าสาขาเซ็นทรัลเวิลด์\nทุกอย่างอร่...,Positive,0.999520,1.0,1.0,0.0,1.0,1.0,0.0,0.0
3,Ci9DQUlRQUNvZENodHljRjlvT2tab09XWkdjR0pyWTBkMm...,Princess Mia,3,ไม่ได้กินบาร์บีก้อนนานแล้ว วันนี้เห็นมี lunch ...,Negative,0.999632,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ChdDSUhNMG9nS0VJQ0FnSURmcjRPMnNnRRAB,Jira Tong,5,Review: 🇹🇭 | 🇬🇧\n\nส่วนตัวไม่ค่อยได้มากิน BarB...,Neutral,0.997700,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,ChZDSUhNMG9nS0VJQ0FnSURJMXJERUN3EAE,Camila K,3,Overpriced and small portions as compared to p...,Negative,0.999206,0.0,0.0,1.0,0.0,0.0,0.0,0.0
96,ChdDSUhNMG9nS0VJQ0FnSUNVX0kzdzhnRRAB,Kussjung Love,4,ทานที่นี่ดีกว่าไปทานในห้างค่ะ คนไม่เยอะ แถมนั่...,Positive,0.998901,1.0,1.0,0.0,1.0,1.0,1.0,0.0
97,ChZDSUhNMG9nS0VJQ0FnSURzbk1maENREAE,Karina Suvapataya,5,Loved the buffet! Want to go back for more. Be...,Neutral,0.908327,0.0,1.0,0.0,0.0,0.0,0.0,1.0
98,ChdDSUhNMG9nS0VJQ0FnSUQybXYtQ3RBRRAB,ยอด สุดสวย,1,รายการโปร ไม่รับ walk in ต้องจองล่วงหน้า แต่ไม...,Negative,0.999191,1.0,0.0,1.0,0.0,0.0,1.0,0.0


# STEP 3: Aggregation

In [32]:
mapping_df = test_df[['sentiment'] + candidate_labels].copy()
aggregate_df = mapping_df.groupby('sentiment').sum().transpose().copy()

In [33]:
aggregate_df

sentiment,Negative,Neutral,Positive
Taste,13.0,8.0,29.0
Portion,6.0,7.0,17.0
Price,15.0,5.0,13.0
Quality,16.0,11.0,35.0
Service,31.0,10.0,33.0
Speed,2.0,2.0,3.0
Location,13.0,8.0,22.0


---

# STEP 4: Priority Matrix

In [34]:
weights = {
	'Taste': 0.30,
	'Quality': 0.25,
	'Price': 0.15,
	'Service': 0.10,
	'Speed': 0.10,
	'Location': 0.05,
	'Portion': 0.05
}

In [35]:
def calculate_aspect_score(row):
	"""
	Converts Neg/Neu/Pos counts into a 0-10 score.
	Pos = 10, Neu = 5, Neg = 0
	"""
	total_mentions = row['Positive'] + row['Neutral'] + row['Negative']
	
	if total_mentions == 0:
		return 0.0 # Avoid division by zero if no mentions
	
	# Weighted sum of sentiments
	score_sum = (row['Positive'] * 10) + (row['Neutral'] * 5) + (row['Negative'] * 0)
	
	return score_sum / total_mentions

In [36]:
# 3. Apply the function to get a score (0-10) for each row
aggregate_df['Aspect_Score'] = aggregate_df.apply(calculate_aspect_score, axis=1)

# 4. Map the importance weights to the dataframe
aggregate_df['Weight'] = aggregate_df.index.map(weights)

# 5. Calculate final contribution (Score * Weight)
aggregate_df['Weighted_Contribution'] = aggregate_df['Aspect_Score'] * aggregate_df['Weight']

# 6. Final Result
final_review_score = aggregate_df['Weighted_Contribution'].sum()

# --- DISPLAY ---
print(aggregate_df[['Aspect_Score', 'Weight', 'Weighted_Contribution']])
print("-" * 30)
print(f"Final Thai Buffet Score: {final_review_score:.2f} / 10")

sentiment  Aspect_Score  Weight  Weighted_Contribution
Taste          6.600000    0.30               1.980000
Portion        6.833333    0.05               0.341667
Price          4.696970    0.15               0.704545
Quality        6.532258    0.25               1.633065
Service        5.135135    0.10               0.513514
Speed          5.714286    0.10               0.571429
Location       6.046512    0.05               0.302326
------------------------------
Final Thai Buffet Score: 6.05 / 10


---

# STEP 5: Text Generation